# 020 — Disaggregation setup & analyses

Sets up and runs the **IML-based** seismic hazard disaggregations for the WP1 sites.

1. Loads the target intensity levels (IMLs) chosen in `017-disagg_imls_for_msa_stripes.ipynb`.
2. Writes one OpenQuake job `.ini` per (IM definition × truncation level × IML).
3. Launches each calculation, **reusing** any whose inputs have not changed.
4. Records every `calc_id` in `wp1/disagg_manifest.json` so the datastores can be
   obtained programmatically later.

## Prerequisites

`003-psha_setup_and_analyses.ipynb` **must have been run first**. This notebook copies
nothing — it reuses what already sits in `hazard_models/eshm20/wp1/`:

| File | Provenance |
|---|---|
| `source_model_logic_tree_eshm20.xml` | untouched ESHM20, placed manually |
| `source_models/` | untouched ESHM20, placed manually |
| `gmpe_logic_tree_AvgSA_0to{3,6}_median_branch.xml` | simplified by hand (single median branch per TRT) |
| `site_model_all_sites.csv` | written from `/results` by notebook 003 |

The target IMLs come from `data_processed/03_site_hazard/AvgSA_{03,06}_imls_for_disaggregation.csv`.
**The AvgSA 0–6 file does not exist yet** — that IM definition is skipped with a warning
until it is created, and picked up automatically once it is.

## Why one config per IML

OpenQuake's `iml_disagg` takes exactly **one** level per IMT, so a separate job is needed
for each target IML. `intensity_measure_types_and_levels` must be absent (the IMTLs are
inferred from `iml_disagg`), and `poes_disagg` cannot be set at the same time.

Each job is **standalone**: it runs its own classical pre-calculation at the single IML
rather than chaining off the PSHA `calc_id`s from 003. The parent PSHA has 25 intensity
levels where these have 1, and that mismatch would break the realization getters.

## Dependencies

**Upstream:** `003-psha_setup_and_analyses.ipynb` (the `wp1/` model),
`017-disagg_imls_for_msa_stripes.ipynb` (the target IMLs).

**Downstream:** the disaggregation post-processing notebook, which reads `disagg-stats`
out of the datastores via `oq_runner.load_calc_ids(...)`. Post-processing is deliberately
**not** done here.

## Runtime
`num_rlzs_disagg = 0` (all 21 SSC realizations) is **required**, not a tuning knob: the
mean disaggregation is only computed when more than one realization is kept. This makes
each run memory-hungry — OpenQuake refuses to start if it cannot fit the accumulator, so
**run one calculation first and check the reported `AccumDict will require X GB`** before
launching the rest.

`DRY_RUN = True` is the default here for that reason.

In [15]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 0. Setup & parameters

In [16]:
import pandas as pd

from phd_project.config import config
from phd_project.scripts import oq_runner

cfg = config.load_config()

In [17]:
# -----------------------------------------------------------------------------
# PARAMETERS
# -----------------------------------------------------------------------------
WP1_DIR = cfg["hazard_models"]["eshm20_wp1"]
MANIFEST_FP = cfg["hazard_models"]["eshm20_wp1_disagg_manifest"]

# IM period range -> csv of target IMLs. The AvgSA 0-6 file does not exist yet;
# any IM whose file is missing is skipped with a warning.
IML_FILES = {
    "03": cfg["proc_data"]["disagg_imls_AvgSA_03"],
    # "06": cfg["proc_data"]["disagg_imls_AvgSA_06"],   # AvgSA[0,6] is not considered beyond this point
}

# Truncation levels (epsilon) to disaggregate at. One calculation is set up and run
# per (IM definition x truncation level x IML).
TRUNCATION_LEVELS = [4]     # truncation levels 3, and 5 have been excluded

# 0 = keep all realizations. Required, not optional: the mean disaggregation
# (disagg-stats) is only computed when more than one realization is kept. Setting
# this to 1 yields the realization *closest to* the mean, not the mean, and
# produces no disagg-stats output at all.
NUM_RLZS_DISAGG = 0

# DRY_RUN:     write configs and report what would run, but launch nothing.
# FORCE_RERUN: re-run every analysis even when its inputs are unchanged.
# NEW_WINDOW:  give each calculation its own console window so its progress can be
#              watched (Windows only; output is teed to WP1_DIR/logs/<name>.log
#              either way). Runs stay sequential regardless.
DRY_RUN = True
FORCE_RERUN = False
NEW_WINDOW = True
# -----------------------------------------------------------------------------

print(f"wp1 dir:  {WP1_DIR}")
print(f"manifest: {MANIFEST_FP}")
print(f"DRY_RUN={DRY_RUN}, FORCE_RERUN={FORCE_RERUN}, NEW_WINDOW={NEW_WINDOW}")

wp1 dir:  C:\Users\clemettn\Documents\phd\hazard_models\eshm20\wp1
manifest: C:\Users\clemettn\Documents\phd\hazard_models\eshm20\wp1\disagg_manifest.json
DRY_RUN=True, FORCE_RERUN=False, NEW_WINDOW=True


## 1. Load the target IMLs

Written by `017-disagg_imls_for_msa_stripes.ipynb`. An IM definition whose file does not
exist yet is skipped rather than treated as an error — this is the expected state for
AvgSA 0–6.

In [18]:
imls_by_im = {}
for im, fp in IML_FILES.items():
    if not fp.is_file():
        print(f"[skip] AvgSA {im}: no IML file at {fp}")
        continue
    imls_by_im[im] = oq_runner.load_imls(fp)
    print(f"AvgSA {im}: {len(imls_by_im[im])} imls from {fp.name}")
    print(f"           {[oq_runner._iml_str(x) for x in imls_by_im[im]]}")

if not imls_by_im:
    raise FileNotFoundError(
        "No IML files found - run 017-disagg_imls_for_msa_stripes.ipynb first.\n"
        + "\n".join(f"  - {fp}" for fp in IML_FILES.values())
    )

n_analyses = sum(len(v) for v in imls_by_im.values()) * len(TRUNCATION_LEVELS)
print(f"\n-> {n_analyses} calculations "
      f"(AvgSA {sorted(imls_by_im)} x eps {TRUNCATION_LEVELS} x imls)")

AvgSA 03: 13 imls from AvgSA_03_imls_for_disaggregation.csv
           ['0.27', '0.285', '0.31', '0.33', '0.36', '0.4', '0.45', '0.5', '0.55', '0.65', '0.8', '0.95', '1.15']

-> 13 calculations (AvgSA ['03'] x eps [4] x imls)


## 2. Write the disaggregation configs

Written flat into `wp1/` alongside the PSHA configs

In [19]:
analyses = {}
for im, imls in imls_by_im.items():
    for eps in TRUNCATION_LEVELS:
        for iml in imls:
            name = oq_runner.disagg_analysis_name(im, eps, iml)
            analyses[name] = oq_runner.write_disagg_config(
                WP1_DIR / oq_runner.disagg_config_name(im, eps, iml),
                description=oq_runner.disagg_description(im, eps, iml),
                gsim_logic_tree_file=oq_runner.GMPE_LOGIC_TREES[im],
                truncation_level=eps,
                im_upper=oq_runner.im_upper(im),
                iml=iml,
                num_rlzs_disagg=NUM_RLZS_DISAGG,
            )

print(f"wrote {len(analyses)} configs to {WP1_DIR}\n")
for name in list(analyses)[:3]:
    print(f"  {name:32s} -> {analyses[name].name}")
print(f"  ... ({len(analyses) - 3} more)")

wrote 13 configs to C:\Users\clemettn\Documents\phd\hazard_models\eshm20\wp1

  AvgSA_03_disagg_eps4_0pt270      -> config_AvgSA_03_disagg_eps4_0pt270.ini
  AvgSA_03_disagg_eps4_0pt285      -> config_AvgSA_03_disagg_eps4_0pt285.ini
  AvgSA_03_disagg_eps4_0pt310      -> config_AvgSA_03_disagg_eps4_0pt310.ini
  ... (10 more)


## 3. Run (or reuse) the calculations

Each analysis is launched only if it has not been run before, or if one of its inputs
(config, logic trees, site model, source models) has changed since it was. Runs are
sequential — the engine parallelises internally, so concurrent calculations would only
contend for cores.

⚠️ **Run a single analysis first** and check the engine's reported memory requirement
before letting this loop through all of them. Set `DRY_RUN = False` when ready.

In [20]:
calc_ids = {}
for name, config_fp in analyses.items():
    calc_ids[name] = oq_runner.run_or_reuse(
        name, config_fp, WP1_DIR, MANIFEST_FP,
        force_rerun=FORCE_RERUN, dry_run=DRY_RUN, new_window=NEW_WINDOW,
    )

calc_ids

[oq] 'AvgSA_03_disagg_eps4_0pt270' would run config_AvgSA_03_disagg_eps4_0pt270.ini (dry_run).
[oq] 'AvgSA_03_disagg_eps4_0pt285' would run config_AvgSA_03_disagg_eps4_0pt285.ini (dry_run).
[oq] 'AvgSA_03_disagg_eps4_0pt310' would run config_AvgSA_03_disagg_eps4_0pt310.ini (dry_run).
[oq] 'AvgSA_03_disagg_eps4_0pt330' would run config_AvgSA_03_disagg_eps4_0pt330.ini (dry_run).
[oq] 'AvgSA_03_disagg_eps4_0pt360' would run config_AvgSA_03_disagg_eps4_0pt360.ini (dry_run).
[oq] 'AvgSA_03_disagg_eps4_0pt400' would run config_AvgSA_03_disagg_eps4_0pt400.ini (dry_run).
[oq] 'AvgSA_03_disagg_eps4_0pt450' would run config_AvgSA_03_disagg_eps4_0pt450.ini (dry_run).
[oq] 'AvgSA_03_disagg_eps4_0pt500' would run config_AvgSA_03_disagg_eps4_0pt500.ini (dry_run).
[oq] 'AvgSA_03_disagg_eps4_0pt550' would run config_AvgSA_03_disagg_eps4_0pt550.ini (dry_run).
[oq] 'AvgSA_03_disagg_eps4_0pt650' would run config_AvgSA_03_disagg_eps4_0pt650.ini (dry_run).
[oq] 'AvgSA_03_disagg_eps4_0pt800' would run confi

{'AvgSA_03_disagg_eps4_0pt270': None,
 'AvgSA_03_disagg_eps4_0pt285': None,
 'AvgSA_03_disagg_eps4_0pt310': None,
 'AvgSA_03_disagg_eps4_0pt330': None,
 'AvgSA_03_disagg_eps4_0pt360': None,
 'AvgSA_03_disagg_eps4_0pt400': None,
 'AvgSA_03_disagg_eps4_0pt450': None,
 'AvgSA_03_disagg_eps4_0pt500': None,
 'AvgSA_03_disagg_eps4_0pt550': None,
 'AvgSA_03_disagg_eps4_0pt650': None,
 'AvgSA_03_disagg_eps4_0pt800': None,
 'AvgSA_03_disagg_eps4_0pt950': None,
 'AvgSA_03_disagg_eps4_1pt150': None}

## 4. Manifest

`wp1/disagg_manifest.json` maps each analysis to its `calc_id`, together with the content
hashes of the inputs that produced it. It is small text and **git-tracked**

Downstream notebooks should read it with `oq_runner.load_calc_ids(MANIFEST_FP)` rather
than hardcoding integers.

In [21]:
import re

manifest = oq_runner.load_manifest(MANIFEST_FP)

if not manifest:
    print(f"no manifest yet at {MANIFEST_FP} (nothing has been run)")
else:
    # AvgSA_<im>_disagg_eps<n>_<iml token, e.g. 0pt285>. Stale index-style keys
    # (..._iml01) from before the rename don't match and are skipped.
    name_re = re.compile(r"^AvgSA_(?P<im>\d+)_disagg_eps(?P<eps>\d+)_(?P<tok>\d+pt\d+)$")
    rows = []
    for name, entry in manifest.items():
        m = name_re.match(name)
        if not m:
            continue
        rows.append({
            "analysis": name,
            "calc_id": entry["calc_id"],
            "im": m["im"],
            "eps": m["eps"],
            "iml": float(m["tok"].replace("pt", ".")),
            "config": entry["config"],
            "written_at": entry["_meta"]["written_at"],
        })
    display(pd.DataFrame(rows)
            .sort_values(["im", "eps", "iml"])
            .reset_index(drop=True))

no manifest yet at C:\Users\clemettn\Documents\phd\hazard_models\eshm20\wp1\disagg_manifest.json (nothing has been run)
